# MediBot — Milestone 3: LLM providers + 4 agent tools

In this notebook we wire together the pieces the orchestrator (Milestone 4) will use. Before we build the ReAct loop, we need to (a) be able to call *some* LLM, and (b) have a set of well-defined tools it can invoke.

## Concepts in play

- **Provider abstraction.** A `LLMProvider` interface with two backends (Gemini = cloud, Ollama = local llama.cpp). Downstream code never knows which one it's talking to — it just calls `.generate()` or `.chat()`. Flipping an env var swaps the model.
- **Pydantic schemas.** Every agent tool has typed input and output via `pydantic.BaseModel`. This gives runtime validation, autogenerated JSON schema (for advertising tools to an LLM), and IDE support.
- **Agent vs. tool.** In this project, the four 'specialized agents' from the brief are actually specialized *tools*. They're deterministic (severity, description, precaution) or single-call (diagnosis). The *agent* is the orchestrator (M4) — it reasons about which tools to invoke. This is a common realignment when you build a multi-agent system: most so-called agents are tools, and one true agent does the routing.
- **Emergency escalation logic.** The severity tool encodes a hard-coded list of symptoms that skip the normal threshold math and go straight to `urgency="emergency"`. Conservative by design — a medical demo must never under-escalate.


In [1]:
import sys, os, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

from medibot.data import load_medical_data
from medibot.rag.embeddings import LocalEmbedding
from medibot.rag.vector_store import FaissStore
from medibot.agents import (
    DiagnosisAgent, SeverityAgent, DescriptionAgent, PrecautionAgent,
    DiagnosisInput, SeverityInput, DescriptionInput, PrecautionInput,
)

data = load_medical_data('../data/raw')
store = FaissStore(LocalEmbedding('BAAI/bge-small-en-v1.5'))
store.load('../data/embeddings/faiss_bge_small')
print(f'Loaded {len(data.diseases)} diseases and a {store.index.ntotal}-vector index.')


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded 41 diseases and a 41-vector index.


/Users/lkshay/CapstoneProject/src/medibot/rag/embeddings.py:45: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dim = int(self._model.get_sentence_embedding_dimension())


## 1. Diagnosis agent — FAISS-backed retrieval

The only agent that uses the vector index. It takes a natural-language query, embeds it, pulls the top-k disease documents, and returns candidates with a score and the subset of known symptoms that appear in the query (a cheap signal for the orchestrator to decide if it should ask follow-up questions).


In [2]:
diagnosis = DiagnosisAgent(store, data)

out = diagnosis(DiagnosisInput(query='I have itching, skin rash and patches on my skin', k=3))
for c in out.candidates:
    print(f'  {c.score:.3f}  {c.disease:30s}  matched={c.matched_symptoms}')
print('confident:', out.confident)


  0.748  fungal infection                matched=['itching', 'skin_rash']
  0.712  psoriasis                       matched=['skin_rash']
  0.683  acne                            matched=['skin_rash']
confident: False


## 2. Severity agent — lookup + thresholds + emergency overrides

No LLM needed. Canonical severity weights (1–7) are joined per symptom; the total is bucketed into low/moderate/urgent/emergency. A hard-coded emergency list short-circuits the bucket for critical symptoms.


In [3]:
severity = SeverityAgent(data)

# Normal case
out = severity(SeverityInput(symptoms=['itching', 'skin_rash', 'continuous_sneezing']))
print('Normal:', out.total_weight, '->', out.urgency)

# Emergency override — chest pain alone escalates
out = severity(SeverityInput(symptoms=['itching', 'chest_pain']))
print('Emergency override:', out.total_weight, '->', out.urgency, '(flags=%s)' % out.emergency_flags)
print('Note:', out.note)

# Unknown symptoms are surfaced, not silently dropped
out = severity(SeverityInput(symptoms=['itching', 'tea_craving', 'zebra_stripes_on_skin']))
print('With unknowns:', out.recognized, 'unrecognized:', out.unrecognized)


Normal: 8 -> moderate
Emergency override: 8 -> emergency (flags=['chest_pain'])
Note: One or more symptoms indicate a potential emergency. Seek immediate medical attention or call emergency services.
With unknowns: {'itching': 1} unrecognized: ['tea_craving', 'zebra_stripes_on_skin']


## 3. Description + Precaution — lookup tools

Both use the alias table from `data.py` so minor misspellings (the hemorrhoids case from EDA) still resolve.


In [4]:
description = DescriptionAgent(data)
precaution = PrecautionAgent(data)

d_out = description(DescriptionInput(disease='Fungal infection'))
print('Description:', d_out.description[:140] + '...')

p_out = precaution(PrecautionInput(disease='Dimorphic hemorrhoids(piles)'))  # intentionally wrong spelling
print(f'\nAlias resolved to: {p_out.disease}')
for i, prec in enumerate(p_out.precautions, 1):
    print(f'  {i}. {prec}')


Description: In humans, fungal infections occur when an invading fungus takes over an area of the body and is too much for the immune system to handle. F...

Alias resolved to: dimorphic hemmorhoids(piles)
  1. avoid fatty spicy food
  2. consume witch hazel
  3. warm bath with epsom salt
  4. consume alovera juice


## 4. LLM providers — one interface, two backends

`get_llm()` reads `LLM_PROVIDER` env var. Passing `"ollama"` below overrides it for demo purposes; in the app we'd just export the env var.

The point of the abstraction: tomorrow we can deploy with Gemini (faster for public demo) and dev locally with Gemma4 (private, free), **with no code changes** downstream.


In [5]:
os.environ['OLLAMA_MODEL'] = 'gemma4:latest'
from medibot.llm import get_llm
from medibot.llm.base import GenerationConfig, Message

llm = get_llm('ollama')
print('Provider:', llm.name)
response = llm.generate(
    'In one sentence, explain what causes a fungal skin infection.',
    system='You are a concise medical assistant. Respond in one sentence.',
)
print(response)


Provider: ollama:gemma4:latest


Fungal skin infections are caused by an overgrowth or colonization of fungi, such as *Candida* or *Trichophyton*, which typically thrive in warm, moist environments.


In [6]:
# Gemini path — only runs if GOOGLE_API_KEY is set.
if os.environ.get('GOOGLE_API_KEY'):
    gemini = get_llm('gemini')
    print('Provider:', gemini.name)
    print(gemini.generate(
        'In one sentence, explain what causes a fungal skin infection.',
        system='You are a concise medical assistant.',
    ))
else:
    print('GOOGLE_API_KEY not set — skipping Gemini test. (Set it in .env to enable.)')


GOOGLE_API_KEY not set — skipping Gemini test. (Set it in .env to enable.)


## What we have now → Milestone 4

- Four typed, testable tools (diagnosis, severity, description, precaution)
- An LLM interface with two interchangeable backends
- Emergency escalation that will override the orchestrator's own judgement

**M4 builds the ReAct orchestrator** — the actual agent. It receives a user turn, decides which tools to invoke (possibly several in sequence), threads their outputs back as observations, and finally produces a grounded answer. We'll build it **from scratch first** so you understand the Reason→Act→Observe loop, then wrap the same logic in LangGraph for the production version.
